# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nadim7009/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Lane 4, CTR opportunity scoring. Task type: ranking, built from a regression score.**

It works in two steps:

1. **Regression:** estimate the CTR a page *should* get, given its position, impressions, search intent, content type and site (client). Call this the expected CTR.
2. **Ranking:** score each page by its click gap = (expected CTR − actual CTR) × impressions, then sort. The top of the list goes to an editor.

Why not the other types:

- **Classification** would need a yes/no truth label like "this title is bad", and no such label exists in the data. Making one up by rule would mean the model just learns my rule.
- **Clustering** would describe groups of pages but wouldn't say which page to fix first.

The editor's question is "which page first?", which is a ranking question. The regression step exists so that "below expectation" is measured fairly rather than guessed.

In [1]:
# Setup: works in Colab (clones my repo) and locally (walks up to the repo root)
import os, sys, subprocess
if "google.colab" in sys.modules:
    if not os.path.isdir("flyrank-ml-internship"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/nadim7009/flyrank-ml-internship"], check=True)
    os.chdir("flyrank-ml-internship")
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd, numpy as np
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 20)
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Lane slice: pages with real position data (avg_position = 0 means "no data"),
# on the first two result pages, with enough impressions for CTR to mean something
pool = df[(df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["impressions_90d"] >= 500)].copy()
pool = pool.reset_index(drop=True)
print(f"Lane slice: {len(pool):,} pages from {pool['client_id'].nunique()} clients "
      f"(out of {len(df):,} pages in the starter data)")

Lane slice: 12,023 pages from 28 clients (out of 30,000 pages in the starter data)


## 2. Target or proxy

**Regression target: `ctr`, the page's measured CTR over 90 days** (clicks ÷ impressions × 100, so 0.24 means 0.24%). This is an **observed outcome**, not a label made by a rule, which is what the framing skill asks for.

**The ranking score is derived from it:** `click_gap = (expected_ctr − ctr) / 100 × impressions_90d`. It's the clicks a page "misses" compared with similar pages. It's a ranking device, not a promise of clicks to win back.

**What I'd validate against (warehouse, later weeks): an observed future outcome.** I'd flag pages using an earlier 90-day window, then check the next 30 days. Does the gap persist, and does CTR stay low? This keeps the feature window and the outcome window apart.

**A proxy flag for inspection only:** `under_capturing = ctr < 0.5 × expected_ctr`. It's defined by a rule, so I'll use it to read examples and count pages, never as a training label.

The cell below sketches these columns, using the simplest expected CTR (the median of the page's position tier). The model in section 5 is meant to replace that.

In [2]:
# Sketch of the target columns, using the simplest possible expected CTR: the median of the page's position tier
pool["expected_ctr"] = pool.groupby("position_tier")["ctr"].transform("median")
pool["click_gap"] = ((pool["expected_ctr"] - pool["ctr"]).clip(lower=0) / 100 * pool["impressions_90d"]).round(1)
pool["under_capturing"] = pool["ctr"] < 0.5 * pool["expected_ctr"]   # inspection only, never a training label

cols = ["position_tier", "avg_position", "impressions_90d", "clicks_90d", "ctr", "expected_ctr",
        "click_gap", "under_capturing"]
print(pool.sort_values("click_gap", ascending=False)[cols].head(8).to_string(index=False))
print(f"\nPages flagged under_capturing: {pool['under_capturing'].sum():,} of {len(pool):,} "
      f"({pool['under_capturing'].mean():.0%})")

position_tier  avg_position  impressions_90d  clicks_90d  ctr  expected_ctr  click_gap  under_capturing
       page_1           7.3           295097         154 0.05          0.24      560.7             True
       page_1           4.2           517715         741 0.14          0.24      517.7            False
       page_1           9.7           208678           0 0.00          0.24      500.8             True
       page_1           7.8           223271          70 0.03          0.24      468.9             True
        top_3           2.3           272144          75 0.03          0.20      462.6             True
       page_1           7.6           140079          16 0.01          0.24      322.2             True
       page_1           4.7           213963         211 0.10          0.24      299.5             True
       page_1           7.8           159590         100 0.06          0.24      287.3             True

Pages flagged under_capturing: 3,031 of 12,023 (25%)


## 3. Success metric

There are two parts, one per step, and both are named before any training.

**Step 1, expected CTR: impressions-weighted mean absolute error (MAE) of CTR on held-out clients.**

- "Weighted by impressions" means pages that are seen more count more, matching what the editor cares about.
- "Held-out clients" means the model is tested on sites it never saw in training.
- **Good** = lower error than the tier-median rule on most client folds. If the model can't beat a lookup table, I use the lookup table.

**Step 2, the review queue: precision@50 against the future outcome.** Of the top 50 pages, how many are still under-capturing in the next window? 50 is about what an editor can review in a week or two.

- **Good** = clearly above the base rate. A random queue would score roughly the share of pages that are under-capturing (about 25% here), so a queue near 0.25 adds nothing.
- I can only measure this on the warehouse, because the starter data has a single 90-day window and no "next window". The cell below works out the bar I need to clear.

In [3]:
# The bar step 2 has to clear: what a random top-50 would score
base_rate = pool["under_capturing"].mean()
print(f"Base rate of under_capturing in the slice: {base_rate:.2f}")
print(f"Expected precision@50 of a random queue: about {base_rate:.2f} "
      f"(~{base_rate * 50:.0f} of 50 pages)")

# A reminder of why held-out clients matter: page counts per client are very uneven
counts = pool["client_id"].value_counts()
print(f"\nLargest client holds {counts.iloc[0] / len(pool):.0%} of the slice; "
      f"median client has {int(counts.median())} pages")

Base rate of under_capturing in the slice: 0.25
Expected precision@50 of a random queue: about 0.25 (~13 of 50 pages)

Largest client holds 37% of the slice; median client has 95 pages


## 4. The unit of analysis, as a real dataframe

**One row = one page (`content_id`), summarising its last 90 days of search data.** That covers its average position, impressions, clicks and CTR, plus context columns (intent, content type, age, word count).

- `client_id` stays on each row only for grouping and for client-held-out splits. It's never a feature value in its own right.
- `trend_direction` and `trend_pct` aren't used anywhere. They're the starter label's source, and my lane doesn't need them.

The cell checks the grain (no duplicate pages) and shows real rows.

In [4]:
# Grain check: one row per content_id
assert pool["content_id"].is_unique, "duplicate pages in the slice"
print(f"Grain OK: {len(pool):,} rows, {pool['content_id'].nunique():,} unique content_id values\n")

unit_cols = ["content_id", "client_id", "position_tier", "avg_position", "impressions_90d", "clicks_90d",
             "ctr", "main_intent", "content_type", "content_age_days", "word_count"]
pool[unit_cols].head(5)

Grain OK: 12,023 rows, 12,023 unique content_id values



             content_id          client_id position_tier  avg_position  impressions_90d  clicks_90d   ctr    main_intent     content_type  content_age_days  word_count
0  content_304f48230142  client_f369cb89fc      striking          10.6             3803          29  0.76  transactional  keyword article               187      3221.0
1  content_331d6c4de07b  client_19581e27de        page_1           6.2            11751          58  0.49     commercial  keyword article               463         NaN
2  content_d4084a4bc775  client_f369cb89fc        page_1           8.5             3970           1  0.03  transactional  keyword article               147      3080.0
3  content_c27558df2b0c  client_19581e27de        page_1           4.9             1240           2  0.16  informational  keyword article               257         NaN
4  content_d8ee6cc6d642  client_19581e27de         top_3           2.2            20919         324  1.55     commercial  keyword article               329     

## 5. Why ML beats a fixed rule here

The fixed rule is "compare each page to the median CTR of its position tier". The data says this rule leaves most of the pattern on the table:

- **Position tier explains very little of the CTR spread.** On a log scale, it accounts for about 1.4% of the variation between pages (cell below).
- **The site matters more than the tier.** Knowing the client explains about 7.5%, and client plus tier together about 13%. A page that looks weak against all sites may be normal for its own site, and the reverse.
- **The factors interact.** Intent, content type and exact position (not just the tier bucket) all shift what "normal" is, and each combination is too thin for a hand-written table. A model can learn those shifts from many pages at once, so the gap gets measured against a fairer expectation.

**Honest first result.** A quick gradient-boosting model, tested on held-out clients, gets a weighted CTR error of about 0.245 against the rule's 0.255. That's roughly 4% better on average, and not better on every fold. So ML isn't clearly winning yet on this small slice. Most of the spread is still unexplained, likely by things the starter data doesn't have, such as the query mix behind each page.

That's the work for the coming weeks: add query-mix features from the warehouse's query table, and keep the rule as the baseline to beat. If the model still can't beat the rule, the honest output is the rule plus reason codes.

In [5]:
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import HistGradientBoostingRegressor

# How much of the (log) CTR spread does each grouping explain?
pool["log_ctr"] = np.log1p(pool["ctr"])
def share_explained(by):
    group_mean = pool.groupby(by)["log_ctr"].transform("mean")
    return 1 - ((pool["log_ctr"] - group_mean) ** 2).sum() / ((pool["log_ctr"] - pool["log_ctr"].mean()) ** 2).sum()
for by in ["position_tier", "client_id", ["client_id", "position_tier"]]:
    print(f"Share of CTR spread explained by {str(by):35s}: {share_explained(by):.1%}")

# Rule vs a first model, tested on clients the model never saw (seed 42)
feats = ["avg_position", "impressions_90d", "content_type", "main_intent", "word_count", "content_age_days"]
X = pool[feats].copy()
for c in ["content_type", "main_intent"]:
    X[c] = X[c].fillna("unknown").astype("category")
y, w = pool["ctr"].values, pool["impressions_90d"].values

rule_err, model_err = [], []
for tr, te in GroupKFold(n_splits=5).split(X, y, groups=pool["client_id"]):
    tier_median = pool.iloc[tr].groupby("position_tier")["ctr"].median()
    rule_pred = pool.iloc[te]["position_tier"].map(tier_median).fillna(pool.iloc[tr]["ctr"].median()).values
    model = HistGradientBoostingRegressor(loss="absolute_error", max_iter=300, learning_rate=0.05,
                                          categorical_features="from_dtype", random_state=42)
    model.fit(X.iloc[tr], y[tr], sample_weight=w[tr])
    rule_err.append(np.average(np.abs(y[te] - rule_pred), weights=w[te]))
    model_err.append(np.average(np.abs(y[te] - model.predict(X.iloc[te])), weights=w[te]))

print("\nImpressions-weighted CTR error on held-out clients (lower is better)")
print(f"  tier-median rule: {np.round(rule_err, 3)}  mean {np.mean(rule_err):.3f}")
print(f"  first model:      {np.round(model_err, 3)}  mean {np.mean(model_err):.3f}")
print(f"  model better on {sum(m < r for m, r in zip(model_err, rule_err))} of 5 folds")

Share of CTR spread explained by position_tier                      : 1.4%
Share of CTR spread explained by client_id                          : 7.5%
Share of CTR spread explained by ['client_id', 'position_tier']     : 13.2%

Impressions-weighted CTR error on held-out clients (lower is better)
  tier-median rule: [0.19  0.377 0.19  0.235 0.285]  mean 0.255
  first model:      [0.196 0.341 0.19  0.223 0.275]  mean 0.245
  model better on 4 of 5 folds


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**The action this supports:** an SEO editor reviews the top pages of the queue and decides whether to rewrite the title and meta description, check the page against the search intent, or leave it and monitor. The queue orders the reading; the editor makes the call.